# Lesson 5 — Time Series: From Canonicals to Testable Signals

**Learning objective:** Understand how the canonical table is converted into a quarterly mention-count matrix and aligned with earnings surprises. Explore what sparsity looks like, why it matters, and how the aligned dataset feeds the Granger analysis.

---

## The core question

We have a canonical mechanism: *"Red Sea logistics disruption suppresses margins"*. Management mentioned it in Q3 2024. The company then reported an earnings miss in Q4 2024. Did the mention *predict* the miss?

To test this we need to convert "mentions" into a time series — one observation per quarter — and line it up with the earnings surprise series.

```
Quarter     | mentions of 'logistics disruption' | earnings surprise (%)
2023-Q1     |  0                                 |  +2.1
2023-Q2     |  1                                 |  -0.3
2023-Q3     |  0                                 |  +1.8
2024-Q1     |  2                                 |  -4.2   ← mention spike before miss
2024-Q2     |  3                                 |  -6.1
2024-Q3     |  2                                 |  -3.8
```

In [ ]:
import sys, json, tempfile
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, '../../src')

from signal_intelligence.timeseries import (
    _quarter_end, build_mention_matrix, align_with_earnings, build_timeseries
)
from ptc_extraction.io import append_to_pool
from ptc_extraction.schema import PTC, Direction, Polarity, SourceType

print("Imports OK")

## Exercise 1 — Quarter-end mapping

In [ ]:
test_dates = [
    ("2024-01-15", "Q1 → 2024-03-31"),
    ("2024-04-01", "Q2 → 2024-06-30"),
    ("2024-09-30", "Q3 → 2024-09-30"),
    ("2024-11-14", "Q4 → 2024-12-31"),
]

for date_str, expected_label in test_dates:
    ts = pd.Timestamp(date_str)
    qe = _quarter_end(ts)
    print(f"  {date_str}  →  quarter end: {qe.date()}   [{expected_label}]")

## Exercise 2 — Build a mention matrix from a PTC pool

In [ ]:
def make_ptc(source_id, source_date, mechanism, direction="precursor", polarity=-1):
    return PTC(
        mechanism=mechanism,
        raw_text=mechanism[:100],
        direction=direction,
        polarity=polarity,
        source_date=source_date,
        source_type=SourceType.MANAGEMENT,
        source_id=source_id,
        extracted_by="test",
        extraction_version="v1.1",
    )


with tempfile.TemporaryDirectory() as tmpdir:
    pool_path = Path(tmpdir) / "pool.jsonl"

    # Logistics mechanism mentioned in Q1, Q2, Q3 2024
    # Forex mechanism mentioned only in Q3 2024
    ptcs = [
        make_ptc("ML.PA-management-2024-01-15-doc1", date(2024, 1, 15),
                 "Red Sea disruptions elevate logistics costs and pressure margins significantly."),
        make_ptc("ML.PA-management-2024-04-20-doc2", date(2024, 4, 20),
                 "Continued Red Sea disruptions maintain elevated freight costs into Q2."),
        make_ptc("ML.PA-management-2024-07-18-doc3", date(2024, 7, 18),
                 "Red Sea situation persists, logistics headwind expected through year end."),
        make_ptc("ML.PA-management-2024-07-18-doc3b", date(2024, 7, 18),
                 "Euro appreciation compressed Q3 revenue translation by approximately 120M euros.",
                 direction="consequence"),
    ]
    append_to_pool(pool_path, ptcs)

    # Canonical table: two canonicals
    canonical_df = pd.DataFrame([
        {"canonical_id": "logistics", "ticker": "ML.PA", "corpus": "management",
         "direction": "precursor", "polarity": -1, "corpus_presence": "insider_only",
         "sector_tag": "untagged", "n_ptcs": 3, "is_noise": False,
         "representative_mechanism": "Red Sea logistics disruption",
         "source_ids": json.dumps([
             "ML.PA-management-2024-01-15-doc1",
             "ML.PA-management-2024-04-20-doc2",
             "ML.PA-management-2024-07-18-doc3",
         ]),
         "matched_canonical_id": None, "cross_map_similarity": None},
        {"canonical_id": "forex", "ticker": "ML.PA", "corpus": "management",
         "direction": "consequence", "polarity": -1, "corpus_presence": "insider_only",
         "sector_tag": "untagged", "n_ptcs": 1, "is_noise": False,
         "representative_mechanism": "Euro translation loss Q3",
         "source_ids": json.dumps(["ML.PA-management-2024-07-18-doc3b"]),
         "matched_canonical_id": None, "cross_map_similarity": None},
    ])

    matrix = build_mention_matrix(canonical_df, pool_path)

print("Mention matrix (quarters × canonicals):")
print(matrix)

## Exercise 3 — Align with earnings

In [ ]:
# Synthetic earnings surprise series — 8 quarters
earnings_dates = pd.date_range("2022-03-31", periods=8, freq="QE")
earnings_df = pd.DataFrame(
    {
        "reported_eps":  [1.2, 1.3, 1.1, 1.4, 1.0, 0.9, 0.8, 1.1],
        "estimated_eps": [1.1, 1.2, 1.2, 1.3, 1.2, 1.1, 1.0, 1.0],
        "surprise_pct":  [9.1, 8.3, -8.3, 7.7, -16.7, -18.2, -20.0, 10.0],
    },
    index=earnings_dates,
)
earnings_df.index.name = "date"

aligned = align_with_earnings(matrix, earnings_df)

print("Aligned dataset (canonical × quarter × surprise):")
print(aligned.to_string())

## Sparsity — the silent killer of time series analysis

A canonical mentioned in only 2 out of 22 quarters has 90% zeros. Granger causality on such a series is meaningless — you're testing whether silence predicts something. The sparsity filter (30% threshold) removes these canonicals before statistical testing.

In [ ]:
# Simulate a sparse canonical (mentioned in only 2/22 quarters)
quarters = pd.date_range("2019-03-31", periods=22, freq="QE")
sparse_mentions = pd.Series([0]*20 + [1, 2], index=quarters, name="sparse_mechanism")
dense_mentions  = pd.Series([0, 1, 2, 1, 0, 3, 2, 1, 0, 2, 1, 3, 2, 1, 0, 2, 3, 1, 2, 1, 0, 1],
                             index=quarters, name="dense_mechanism")

SPARSITY_THRESHOLD = 0.30

for series in [sparse_mentions, dense_mentions]:
    nonzero_frac = (series > 0).mean()
    action = "KEEP" if nonzero_frac >= SPARSITY_THRESHOLD else "FILTER OUT"
    print(f"{series.name:25s}  non-zero: {nonzero_frac:.1%}  → {action}")

## Visualise: mentions alongside earnings surprises

In [ ]:
try:
    import matplotlib.pyplot as plt

    logistics_aligned = aligned[aligned["canonical_id"] == "logistics"].set_index("quarter_end")

    if not logistics_aligned.empty:
        fig, ax1 = plt.subplots(figsize=(10, 4))
        ax2 = ax1.twinx()

        ax1.bar(logistics_aligned.index, logistics_aligned["mention_count"],
                color="steelblue", alpha=0.6, label="Mention count")
        ax2.plot(logistics_aligned.index, logistics_aligned["surprise_pct"],
                 color="firebrick", marker="o", linewidth=1.5, label="Earnings surprise (%)")

        ax1.set_ylabel("Mention count", color="steelblue")
        ax2.set_ylabel("Surprise (%)", color="firebrick")
        ax1.set_title("Logistics disruption: mentions vs earnings surprise")

        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

        plt.tight_layout()
        plt.show()
    else:
        print("No overlapping quarters for logistics canonical — try widening the earnings date range.")
except ImportError:
    print("matplotlib not installed — skipping visualisation.")

---

## Key takeaways

1. `source_date` (not `event_date`) determines which quarter a PTC falls into. Using `event_date` would be forward-looking bias.
2. The mention matrix is sparse by nature — most mechanisms appear in fewer than half the quarters.
3. The 30% sparsity filter is non-negotiable: testing Granger on a mostly-zero series produces meaningless results.
4. The aligned dataset is the direct input to Granger analysis (Track A) — one observation per (canonical, quarter).

**Next:** Lesson 6 — Statistical Tests: Granger causality and event studies.